<center>
<img src="https://laelgelcpublic.s3.sa-east-1.amazonaws.com/lael_50_years_narrow_white.png.no_years.400px_96dpi.png" width="300" alt="LAEL 50 years logo">
<h3>APPLIED LINGUISTICS GRADUATE PROGRAMME (LAEL)</h3>
</center>
<hr>

# Corpus Linguistics - Study 1 - Phase 1 - Julio

The aim of this phase is to:

- Prepare the CEP corpus for tagging with Biber Tagger;
- Extract Biber's (1988) Dimension Scores for each text from the Biber Tag Count output file.


## Organise the corpus
- Parse the text files in `corpus/01_CEP` recursively.
- Each file has a well-defined internal structure. For instance, the `CEP_coding_0001.txt` file has the following structure:

```
<file CEP_coding_0001> <domain CODING> <corpus CEP_Dias_2026>

[88170]
I have a dataset with dates as columns and rows containing city and number of outward flights on that particular date given in the corresponding column. How can I plot this information in map?

[20624]
Write a short JIRA issue description about this, mentioning it should be a flow diagram with information about the endpoints concerned. Document how we implemented authentication with Identity Server with a diagram.

[33356]
What does this method do? Code_Snippet.
```

- For each file, capture each text preceded by a numeric `text_id` in square brackets `[text_id]` and store it in a text file named after the filename plus the `text_id`. For instance, if the filename is `CEP_coding_0001.txt` and the `text_id` is `88170`, the text file will be named `CEP_coding_0001_88170.txt`.
- Save each text file in the `corpus/02_CEP_organised` directory. `text_id`s must be unique within each file. Raise an error if a duplicate `text_id` is found and save the duplicate file with a suffix `_duplicate_[n]`, where `[n]` is the number of times the `text_id` has been found in the file.

### Task (from previous cell)
Parse all `.txt` files under `corpus/01_CEP` recursively. For each file, extract every text that starts with a unique numeric `[text_id]` marker, and write it to `corpus/02_CEP_organised` as `{original_filename_without_ext}_{text_id}.txt`. If a `text_id` repeats within the same source file, raise an error and still save the repeated instances using suffix `{...}_{text_id}_duplicate_[n].txt`.

In [1]:
from __future__ import annotations

from pathlib import Path
import re

SRC_DIR = Path("corpus") / "01_CEP"
DST_DIR = Path("corpus") / "02_CEP_organised"

TEXT_ID_LINE_RE = re.compile(r"^\[(\d+)\]\s*$")


def _iter_text_blocks(file_text: str):
    """
    Yield (text_id:str, block_text:str) in the order they appear.
    A block starts at a line exactly like: [12345]
    The block text is everything after that line up to (but not including)
    the next [text_id] line or EOF. Leading/trailing whitespace is stripped.
    """
    lines = file_text.splitlines()
    starts: list[tuple[int, str]] = []

    for i, line in enumerate(lines):
        m = TEXT_ID_LINE_RE.match(line.strip())
        if m:
            starts.append((i, m.group(1)))

    for idx, (start_i, text_id) in enumerate(starts):
        content_start = start_i + 1
        content_end = starts[idx + 1][0] if idx + 1 < len(starts) else len(lines)
        block = "\n".join(lines[content_start:content_end]).strip()
        yield text_id, block


def organise_cep_corpus(src_dir: Path = SRC_DIR, dst_dir: Path = DST_DIR) -> dict[str, int]:
    """
    Returns a summary dict with counts.
    Raises ValueError if any duplicate text_id is found within a source file,
    but still writes duplicate outputs with _duplicate_[n] suffix.
    """
    if not src_dir.exists():
        raise FileNotFoundError(f"Source directory not found: {src_dir.resolve()}")
    dst_dir.mkdir(parents=True, exist_ok=True)

    summary = {
        "files_scanned": 0,
        "source_txt_files": 0,
        "texts_found": 0,
        "texts_written": 0,
        "files_with_duplicates": 0,
        "duplicate_text_ids_total": 0,
    }

    duplicate_messages: list[str] = []

    for src_path in sorted(src_dir.rglob("*")):
        summary["files_scanned"] += 1
        if not (src_path.is_file() and src_path.suffix.lower() == ".txt"):
            continue

        summary["source_txt_files"] += 1

        raw = src_path.read_text(encoding="utf-8", errors="replace")
        base = src_path.stem  # e.g., CEP_coding_0001

        seen: dict[str, int] = {}  # text_id -> count seen so far (1 means first)
        file_has_dup = False

        for text_id, block in _iter_text_blocks(raw):
            summary["texts_found"] += 1
            seen[text_id] = seen.get(text_id, 0) + 1

            if seen[text_id] == 1:
                out_name = f"{base}_{text_id}.txt"
            else:
                file_has_dup = True
                n = seen[text_id] - 1  # first duplicate is 1
                out_name = f"{base}_{text_id}_duplicate_{n}.txt"
                summary["duplicate_text_ids_total"] += 1

            out_path = dst_dir / out_name
            out_path.write_text(block + "\n", encoding="utf-8")
            summary["texts_written"] += 1

        if file_has_dup:
            summary["files_with_duplicates"] += 1
            dups = sorted([tid for tid, c in seen.items() if c > 1], key=int)
            duplicate_messages.append(
                f"{src_path.name}: duplicate text_id(s) found: {', '.join(dups)}"
            )

    if duplicate_messages:
        raise ValueError(
            "Duplicate text_id(s) detected (duplicates were still written with _duplicate_[n] suffix):\n"
            + "\n".join(duplicate_messages)
        )

    return summary


summary = organise_cep_corpus()
summary


{'files_scanned': 200,
 'source_txt_files': 196,
 'texts_found': 1904,
 'texts_written': 1904,
 'files_with_duplicates': 0,
 'duplicate_text_ids_total': 0}

## Prepare the corpus for tagging with Biber Tagger

In [2]:
import os
import re

# Define input and output directories
input_directory = 'corpus/02_CEP_organised'
output_directory = 'corpus/03_CEP_fixed'

# Create the output directory if it doesn't exist
os.makedirs(output_directory, exist_ok=True)

# Iterate through all files in the input directory
for filename in os.listdir(input_directory):
    if filename.endswith(".txt"):
        input_path = os.path.join(input_directory, filename)
        output_path = os.path.join(output_directory, filename)

        try:
            with open(input_path, 'r', encoding='utf-8') as f:
                # Read all lines to preserve original structure
                original_lines = f.readlines()

            processed_output = []

            for line in original_lines:
                # 1. Insert space between digit and letter (e.g., "1a" -> "1 a")
                fixed_line = re.sub(r'(\d)([a-zA-Z])', r'\1 \2', line)

                # Strip whitespace to process words, but check if line was just empty space
                stripped_line = fixed_line.strip()

                if not stripped_line:
                    # If the original line was empty (e.g. a paragraph break), preserve it
                    processed_output.append("")
                    continue

                # 2. Wrap every 10 words WITHIN the current line
                words = stripped_line.split()

                # Chunk the words into groups of 10
                for i in range(0, len(words), 10):
                    chunk = words[i:i+10]
                    processed_output.append(" ".join(chunk))

            # Join all processed lines with newlines
            final_output = "\n".join(processed_output)

            # Save to the output file
            with open(output_path, 'w', encoding='utf-8') as f:
                f.write(final_output)

            print(f"Processed: {filename}")

        except Exception as e:
            print(f"Error processing {filename}: {e}")

print("All files processed.")

Processed: CEP_editing_0018_82638.txt
Processed: CEP_coding_0009_64939.txt
Processed: CEP_Info_0005_18888.txt
Processed: CEP_Info_0015_18078.txt
Processed: CEP_Info_0042_31345.txt
Processed: CEP_Info_0012_23820.txt
Processed: CEP_Info_0046_62387.txt
Processed: CEP_creative_0008_23532.txt
Processed: CEP_Info_0039_41336.txt
Processed: CEP_creative_0011_71491.txt
Processed: CEP_editing_0030_60085.txt
Processed: CEP_Info_0024_21721.txt
Processed: CEP_editing_0020_84069.txt
Processed: CEP_creative_0035_25262.txt
Processed: CEP_coding_0040_56205.txt
Processed: CEP_creative_0035_45036.txt
Processed: CEP_editing_0037_60789.txt
Processed: CEP_coding_0004_27107.txt
Processed: CEP_creative_0018_28353.txt
Processed: CEP_editing_0051_35234.txt
Processed: CEP_creative_0049_58249.txt
Processed: CEP_Info_0009_49412.txt
Processed: CEP_coding_0021_73412.txt
Processed: CEP_coding_0032_55419.txt
Processed: CEP_creative_0009_77656.txt
Processed: CEP_coding_0034_19156.txt
Processed: CEP_coding_0009_76109.tx

## Extract Biber's (1988) data from the Biber Tag Count output file

The following columns will be extracted for each text in the Biber Tag Count output file:
- Normed Frequency Counts for the Linguistic Features
- Dimension Scores

### Biber Tag Count output file description and sample

This file is composed of records and fields. Each record corresponds to a text in the corpus. Each record is a sequence of 12 lines. Within each record there is a series of fields. In the first line of the record, there are four fields. In the following ten lines, there are 15 fields on each line. And in the last line, there are ten fields. In total, therefore, there are 164 fields.

```
                                  amy_season_10.straight.txt 32.3  4.0 6748
 24.2  8.3 68.9156.9 61.8  4.0 14.5 10.8 77.8 20.9  3.1  2.8  4.7  7.9  3.7
  1.5  8.9 10.7 12.0  2.4  5.3171.8 69.2 20.7 33.3 18.2  5.8  5.8  0.1  1.2
  0.1  0.4 19.9  5.2  8.2 44.9 11.7 11.9  0.9  4.6  2.8  3.3 14.8  2.4  0.6
  1.0  3.4  3.0  0.4  8.6 25.3 23.3  4.0 11.3  1.5 63.6157.8  3.9186.9 18.1
  1.2  1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
  0.0  0.0  0.0  3.7  0.7  3.4  4.0  0.7  0.3  0.1  0.0  0.0  0.1  0.0  0.1
  0.6  5.3  1.3  0.3  0.0  1.0  0.1  0.3  0.0  0.3  0.3  0.0  5.5  2.5  0.0
 11.9  1.2  0.1 13.2  7.7  1.5  9.5  8.3  0.0  0.0  0.0  0.0  0.0  0.0  0.0
  0.9  0.3  0.7  0.0  0.0  0.0  1.0  7.6  4.4  3.1 14.7 10.8  1.8  6.7  5.2
  1.9  1.8  0.1  0.0  1.3  0.6  0.1 31.9 13.0 39.0  3.1  2.5  7.6  4.3  0.0
     48.62     -1.64     -3.10      1.24      6.79  0.1  0.0  0.1  0.1  0.0
                                  amy_season_11.straight.txt 30.0  3.9 7358
 23.6  9.9 64.0154.9 52.7  5.7 13.6 11.6 85.5 22.7  4.1  2.9  4.2  8.4  3.5
  1.8  7.9 10.6 14.5  2.0  4.3173.8 60.1 18.8 33.6 18.3  6.0  7.3  0.3  0.1
  0.1  1.1 14.5  6.8  7.5 45.5 14.4 13.3  0.8  5.7  3.9  3.1 16.9  3.3  0.3
  1.0  3.1  4.1  0.7  7.5 27.9 27.3  4.5  9.9  0.5 56.8156.6  5.0188.9 16.6
  1.9  0.8  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
  0.0  0.0  0.0  3.1  1.4  3.8  6.1  0.8  0.3  0.1  0.0  0.0  0.0  0.0  0.5
  0.5  5.6  2.3  0.1  0.0  0.0  0.1  0.1  0.4  0.3  0.4  0.0  6.0  2.9  0.0
 14.4  1.2  0.0 15.6  9.1  0.7 10.1  9.2  0.0  0.0  0.0  0.0  0.0  0.0  0.0
  0.3  0.7  0.1  0.0  0.0  0.1  0.7  9.2  5.2  2.2 10.5  8.8  2.4  8.2  3.4
  2.2  0.8  0.4  0.1  1.2  1.0  0.3 32.3 14.5 42.0  1.4  2.9  4.9  2.7  0.0
     47.85     -1.30     -3.99      2.55      7.92  0.1  0.0  0.0  0.1  0.0

```

### Biber Tag Count column assignment

|     |       |     |       |     |       |     |       |     |       |      |          |          |        |        |
|-----|-------|-----|-------|-----|-------|-----|-------|-----|-------|------|----------|----------|--------|--------|
|     |       |     |       |     |       |     |       |     |       |      | Filename | Type/Tkn | WrdLen | WrdCnt |
| 1   | 2     | 3   | 4     | 5   | 6     | 7   | 8     | 9   | 10    | 11   | 12       | 13       | 14     | 15     |
| 16  | 17    | 18  | 19    | 20  | 21    | 22  | 23    | 24  | 25    | 26   | 27       | 28       | 29     | 30     |
| 31  | 32    | 33  | 34    | 35  | 36    | 37  | 38    | 39  | 40    | 41   | 42       | 43       | 44     | 45     |
| 46  | 47    | 48  | 49    | 50  | 51    | 52  | 53    | 54  | 55    | 56   | 57       | 58       | 59     | 60     |
| 61  | 62    | 63  | 64    | 65  | 66    | 67  | 68    | 69  | 70    | 71   | 72       | 73       | 74     | 75     |
| 100 | 101   | 102 | 103   | 104 | 105   | 106 | 107   | 108 | 109   | 110  | 111      | 112      | 113    | 114    |
| 115 | 116   | 117 | 118   | 119 | 120   | 121 | 122   | 123 | 124   | 125  | 126      | 127      | 128    | 129    |
| 130 | 131   | 132 | 133   | 134 | 135   | 136 | 137   | 138 | 139   | 140  | 141      | 142      | 143    | 144    |
| 151 | 152   | 153 | 154   | 155 | 156   | 157 | 158   | 159 | 160   | 161  | 162      | 163      | 164    | 165    |
| 166 | 167   | 168 | 169   | 170 | 171   | 172 | 173   | 174 | 175   | 176  | 177      | 178      | 179    | 180    |
|     | Dim_1 |     | Dim_2 |     | Dim_3 |     | Dim_4 |     | Dim_5 | NA_1 | NA_2     | NA_3     | NA_4   | NA_5   |



### Coding task

- Create a robust parser for the **Biber Tag Count** `counts.txt` format (records of **12 lines / 164 fields**):
  - Line 1: `Filename` + `Type/Tkn` + `WrdLen` + `WrdCnt`
  - Lines 2–11: **150** normed frequency values (10 × 15)
  - Line 12: `Dim_1`–`Dim_5` + `NA_1`–`NA_5`
- Return two tidy DataFrames:
  - `df_variables`: filename + metadata + **Var_1..Var_150**
  - `df_dimensions`: filename + **Dim_1..Dim_5**
- Keep filename normalization consistent (e.g., drop `.straight` and `.txt` if present).

In [18]:
import re
import pandas as pd


def _normalize_biber_filename(name: str) -> str:
    name = name.strip()
    name = re.sub(r"\s+", " ", name)
    name = re.sub(r"\.straight(?=\.txt$)", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\.txt$", "", name, flags=re.IGNORECASE)
    return name.lower()


def _parse_fixed_width_numbers(s: str, *, width: int, expected: int, context: str) -> list[float]:
    """
    Parse `expected` numeric fields from string `s` where each field is `width` chars.
    Fields may be space-padded; empty fields are invalid.
    """
    if len(s) < width * expected:
        s = s.ljust(width * expected)

    nums: list[float] = []
    for i in range(expected):
        chunk = s[i * width : (i + 1) * width]
        token = chunk.strip()
        if token == "":
            raise ValueError(f"Empty fixed-width field while parsing {context}. Chunk={chunk!r}")
        try:
            nums.append(float(token))
        except ValueError as e:
            raise ValueError(f"Bad numeric token {token!r} while parsing {context}. Chunk={chunk!r}") from e
    return nums


def _parse_fixed_width_fields_allow_blank(s: str, *, width: int, expected: int, context: str) -> list[str]:
    """
    Parse `expected` fixed-width fields as raw strings (stripped).
    Blank fields are allowed and returned as "".
    """
    if len(s) < width * expected:
        s = s.ljust(width * expected)

    fields: list[str] = []
    for i in range(expected):
        chunk = s[i * width : (i + 1) * width]
        fields.append(chunk.strip())
    return fields


def parse_biber_tagcount_counts(file_path: str, *, keep_original_filename: bool = False) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Parse Biber Tag Count `counts.txt`.

    Record layout (12 lines):
      - Line 1: Filename + 3 numbers (Type/Tkn, WrdLen, WrdCnt)
      - Lines 2-11: 150 numbers (10 lines × 15 fields, each 5 chars wide)
      - Line 12: Dimension/NA layout with spacer fields:
          field 1 blank, field 2 Dim_1, field 3 blank, field 4 Dim_2, ...
          field 10 Dim_5, fields 11-15 NA_1..NA_5
    """
    with open(file_path, "r", encoding="utf-8") as f:
        raw_lines = [ln.rstrip("\n") for ln in f]

    lines = [ln for ln in raw_lines if ln.strip()]
    block_size = 12
    field_width = 5

    var_rows = []
    dim_rows = []

    for base in range(0, len(lines), block_size):
        block = lines[base : base + block_size]
        if len(block) < block_size:
            break

        header = block[0]
        header_parts = header.split()
        if len(header_parts) < 1:
            raise ValueError(f"Missing filename at record starting near line {base + 1}")

        original_filename = header_parts[0]
        filename = original_filename if keep_original_filename else _normalize_biber_filename(original_filename)

        pos = header.find(original_filename)
        if pos == -1:
            raise ValueError(f"Could not locate filename in header at record starting near line {base + 1}: {header!r}")

        header_numeric_tail = header[pos + len(original_filename) :]
        header_nums = _parse_fixed_width_numbers(
            header_numeric_tail,
            width=field_width,
            expected=3,
            context=f"header numerics for {original_filename!r} (record starting near line {base + 1})",
        )

        type_tkn = float(header_nums[0])
        wrdlen = float(header_nums[1])
        wrdcnt = int(round(float(header_nums[2])))

        vars_150: list[float] = []
        for li, ln in enumerate(block[1:11], start=2):
            vars_150.extend(
                _parse_fixed_width_numbers(
                    ln,
                    width=field_width,
                    expected=15,
                    context=f"middle line {li} for {original_filename!r} (record starting near line {base + 1})",
                )
            )

        if len(vars_150) != 150:
            raise ValueError(
                f"Internal parse error: expected 150 vars, got {len(vars_150)} "
                f"for {original_filename!r} (record starting near line {base + 1})"
            )

        # Dimensions line: 15 fixed-width slots (with spacer blanks before each Dim)
        dim_fields = _parse_fixed_width_fields_allow_blank(
            block[11],
            width=field_width,
            expected=15,
            context=f"dimensions line for {original_filename!r} (record starting near line {base + 1})",
        )

        dim_idx = [1, 3, 5, 7, 9]
        na_idx = [10, 11, 12, 13, 14]

        try:
            dims_5 = [float(dim_fields[i]) for i in dim_idx]
        except ValueError as e:
            raise ValueError(
                f"Could not parse Dim_1..Dim_5 from fields {dim_idx} for {original_filename!r}. "
                f"Got={ [dim_fields[i] for i in dim_idx] }"
            ) from e

        try:
            na_5 = [float(dim_fields[i]) for i in na_idx]
        except ValueError as e:
            raise ValueError(
                f"Could not parse NA_1..NA_5 from fields {na_idx} for {original_filename!r}. "
                f"Got={ [dim_fields[i] for i in na_idx] }"
            ) from e

        var_row = [filename, type_tkn, wrdlen, wrdcnt] + vars_150 + na_5
        dim_row = [filename] + dims_5

        var_rows.append(var_row)
        dim_rows.append(dim_row)

    var_columns = (
            ["Filename", "Type/Tkn", "WrdLen", "WrdCnt"]
            + [f"Var_{k}" for k in range(1, 151)]
            + [f"NA_{k}" for k in range(1, 6)]
    )
    dim_columns = ["Filename"] + [f"Dim_{k}" for k in range(1, 6)]

    df_variables = pd.DataFrame(var_rows, columns=var_columns)
    df_dimensions = pd.DataFrame(dim_rows, columns=dim_columns)
    return df_variables, df_dimensions

In [19]:
counts_all_seasons_utf_8 = 'corpus/03_all_seasons_utf_8_tagged/tagcount/counts.txt'

df_variables_all_seasons_utf_8, df_dimensions_all_seasons_utf_8_v2 = parse_biber_tagcount_counts(
    counts_all_seasons_utf_8)

df_dimensions_all_seasons_utf_8_v2


,Filename,Dim_1,Dim_2,Dim_3,Dim_4,Dim_5
0,amy_season_10,48.62,-1.64,-3.10,1.24,6.79
1,amy_season_11,47.85,-1.30,-3.99,2.55,7.92
2,amy_season_12,51.66,-1.60,-3.23,1.86,5.86
3,amy_season_3,18.90,-1.08,-8.36,10.13,-3.63
4,amy_season_4,18.87,-2.39,-0.55,2.11,3.15
...,...,...,...,...,...,...
86,stuart_season_5,28.41,-3.07,-0.04,-0.95,4.17
87,stuart_season_6,43.78,-3.46,-6.01,0.80,1.26
88,stuart_season_7,39.55,-1.62,-4.34,0.88,4.91
89,stuart_season_8,44.77,0.28,-4.94,-0.79,6.24


In [20]:
df_variables_all_seasons_utf_8

,Filename,Type/Tkn,WrdLen,WrdCnt,Var_1,Var_2,Var_3,Var_4,Var_5,Var_6,...,Var_146,Var_147,Var_148,Var_149,Var_150,NA_1,NA_2,NA_3,NA_4,NA_5
0,amy_season_10,32.3,4.0,6748,24.2,8.3,68.9,156.9,61.8,4.0,...,3.1,2.5,7.6,4.3,0.0,0.1,0.0,0.1,0.1,0.0
1,amy_season_11,30.0,3.9,7358,23.6,9.9,64.0,154.9,52.7,5.7,...,1.4,2.9,4.9,2.7,0.0,0.1,0.0,0.0,0.1,0.0
2,amy_season_12,31.5,3.9,6196,24.7,10.0,69.6,167.2,53.7,3.6,...,4.2,1.3,6.1,1.9,0.0,0.5,0.0,0.2,0.0,0.0
3,amy_season_3,18.8,4.0,97,10.3,0.0,61.9,134.0,30.9,10.3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,amy_season_4,36.5,4.4,4846,18.4,6.4,38.8,131.9,42.7,4.1,...,3.1,2.7,5.6,2.3,0.0,0.4,0.2,0.2,0.0,0.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86,stuart_season_5,36.3,3.8,683,19.0,4.4,60.0,146.4,51.2,1.5,...,7.3,0.0,5.9,1.5,0.0,1.5,0.0,0.0,0.0,1.5
87,stuart_season_6,36.0,4.0,767,14.3,6.5,48.2,139.5,52.2,2.6,...,1.3,1.3,7.8,5.2,0.0,1.3,0.0,0.0,0.0,0.0
88,stuart_season_7,31.8,3.7,987,25.3,10.1,62.8,151.0,38.5,4.1,...,4.1,1.0,10.1,4.1,0.0,0.0,0.0,0.0,0.0,0.0
89,stuart_season_8,31.3,3.7,823,20.7,13.4,66.8,132.4,51.0,1.2,...,1.2,0.0,6.1,1.2,0.0,0.0,0.0,0.0,0.0,0.0


In [22]:
df_dimensions_all_seasons_utf_8_v2.shape

(91, 6)

In [23]:
df_variables_all_seasons_utf_8.shape

(91, 159)

### Consolidation task
Create a single consolidated DataFrame by merging `df_variables_all_seasons_utf_8` into `df_dimensions_all_seasons_utf_8_v2` on `Filename`, then sort the result in ascending order by `Filename`.

In [26]:
import pandas as pd

df_consolidated_all_seasons_utf_8 = (
    df_dimensions_all_seasons_utf_8_v2
    .merge(df_variables_all_seasons_utf_8, on="Filename", how="left", validate="one_to_one")
    .assign(
        character=lambda d: d["Filename"].str.extract(r"^([a-z_]+)_season_\d+$", expand=False),
        season=lambda d: pd.to_numeric(d["Filename"].str.extract(r"_season_(\d+)$", expand=False), errors="coerce"),
    )
    .sort_values(["character", "season", "Filename"], ascending=True, kind="mergesort")
    .drop(columns=["character", "season"])
    .reset_index(drop=True)
)

df_consolidated_all_seasons_utf_8


,Filename,Dim_1,Dim_2,Dim_3,Dim_4,Dim_5,Type/Tkn,WrdLen,WrdCnt,Var_1,...,Var_146,Var_147,Var_148,Var_149,Var_150,NA_1,NA_2,NA_3,NA_4,NA_5
0,amy_season_3,18.90,-1.08,-8.36,10.13,-3.63,18.8,4.0,97,10.3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,amy_season_4,18.87,-2.39,-0.55,2.11,3.15,36.5,4.4,4846,18.4,...,3.1,2.7,5.6,2.3,0.0,0.4,0.2,0.2,0.0,0.2
2,amy_season_5,30.42,-2.04,-1.21,1.44,0.51,36.3,4.1,5765,16.8,...,2.8,3.3,3.8,2.4,0.0,0.0,0.0,0.3,0.0,0.0
3,amy_season_6,36.01,-1.37,-3.41,0.73,3.91,34.8,4.0,4888,21.5,...,2.9,1.6,4.9,1.8,0.0,0.4,0.0,0.0,0.2,0.2
4,amy_season_7,43.11,-1.18,-3.23,1.15,2.91,33.5,3.9,5317,20.9,...,1.5,2.3,7.1,2.1,0.0,0.0,0.0,0.2,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86,stuart_season_8,44.77,0.28,-4.94,-0.79,6.24,31.3,3.7,823,20.7,...,1.2,0.0,6.1,1.2,0.0,0.0,0.0,0.0,0.0,0.0
87,stuart_season_9,37.52,-0.58,-2.80,1.79,6.38,31.5,3.7,1280,19.5,...,4.7,3.1,5.5,2.3,0.0,0.8,0.0,0.8,0.0,0.0
88,stuart_season_10,36.93,-1.85,-5.64,2.72,7.05,31.8,3.7,1855,22.6,...,2.2,0.5,8.1,1.1,0.0,0.0,0.0,0.0,0.0,0.0
89,stuart_season_11,51.10,-1.37,-6.39,1.48,5.37,34.8,3.8,1474,22.4,...,2.7,1.4,6.8,1.4,0.0,0.0,0.0,0.0,0.0,0.0


## Exporting to a file

In [27]:
import os
import pandas as pd

# Define output directory
output_directory = 'cl_st1_ph2_marcia'
os.makedirs(output_directory, exist_ok=True)

# Define output filename
filename = 'dimensions_variables_all_seasons_utf_8'

# Export to JSONL
df_consolidated_all_seasons_utf_8.to_json(f"{output_directory}/{filename}.jsonl", orient='records', lines=True)

# Export to Excel
df_consolidated_all_seasons_utf_8.to_excel(f"{output_directory}/{filename}.xlsx", index=False)